## **1. Persiapan dan Scraping Data**

Bagian ini akan menginstall library dan membaca 100 baris data dari file CSV

In [5]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

# Load data
df_links = pd.read_csv('Link Artikel BPJS(Kelompok 8) - Alam.csv')
df_links.columns = df_links.columns.str.strip()

links = df_links['Link'].dropna().tolist()
penerbit = df_links['Penerbit'].dropna().tolist()

def ambil_teks(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        res = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(res.text, 'html.parser')
        paragraphs = soup.find_all('p')
        content = " ".join([p.get_text() for p in paragraphs])
        # Case folding & cleaning
        content = content.lower()
        content = re.sub(r'[^a-z\s]', '', content)
        return content
    except:
        return ""

print("Mengambil konten berita...")
corpus = [ambil_teks(u) for u in links]
print("Selesai.")

Mengambil konten berita...
Selesai.


## **2. Implementasi Vector Space Model**

Di sini kita akan membuat tabel frekuensi kata (Term Frequency) persis seperti di tabel df

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

# Menentukan Vocabulary (Kata kunci spesifik tugas BPJS PBI kamu)
# Di notebook dosen, ini adalah list ['hate', 'darkness', dsb]
vocab_bpjs = ['bpjs', 'pbi', 'data', 'nonaktif', 'kemensos', 'iuran', 'peserta', 'miskin']

# Inisialisasi Vectorizer dengan vocabulary tersebut
vectorizer = CountVectorizer(vocabulary=vocab_bpjs)

# Transformasi corpus menjadi array
vsm_array = vectorizer.fit_transform(corpus).toarray()

# Buat DataFrame VSM (Persis seperti df di notebook dosen)
df_vsm = pd.DataFrame(vsm_array, columns=vocab_bpjs)
df_vsm.index = [f"{p} ({i})" for i, p in enumerate(penerbit)]

print("Tabel Vector Space Model (Frekuensi Kata):")
display(df_vsm)

Tabel Vector Space Model (Frekuensi Kata):


,bpjs,pbi,data,nonaktif,kemensos,iuran,peserta,miskin
Detik (0),0,0,0,0,0,0,0,0
Detik (1),3,6,2,3,1,1,6,0
MetroTV (2),2,2,2,0,4,3,6,2
Antara (3),5,10,5,4,0,2,13,2
Lampost (4),4,6,3,3,3,3,7,0
Detik (5),2,0,0,0,0,1,2,0
Detik (6),7,0,0,0,0,5,6,0
Detik (7),3,0,0,0,0,1,1,0
Tempo (8),20,17,8,2,0,2,4,0
Pemkab Buleleng (9),0,0,0,0,0,0,0,0


## **3. Analisis Cosine Similarity**

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

# Menghitung matriks kemiripan
res_similarity = cosine_similarity(vsm_array)

# Mengubahnya menjadi DataFrame agar mudah dianalisis (Matriks Simetris)
df_similarity = pd.DataFrame(res_similarity,
                             columns=df_vsm.index,
                             index=df_vsm.index)

print("\nMatriks Cosine Similarity (Kedekatan antar Artikel):")
display(df_similarity)


Matriks Cosine Similarity (Kedekatan antar Artikel):


,Detik (0),Detik (1),MetroTV (2),Antara (3),Lampost (4),Detik (5),Detik (6),Detik (7),Tempo (8),Pemkab Buleleng (9),...,Detik (14),Detik (15),Detik (16),Detik (17),Detik (18),Detik (19),Detik (20),TImes Indonesia (21),UNESA (22),Lampost (23)
Detik (0),0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Detik (1),0.0,1.000000,0.756018,0.975418,0.967892,0.646393,0.603337,0.492366,0.768906,0.0,...,0.945108,0.950114,0.928361,0.694973,0.847606,0.751301,0.850386,0.584685,0.896707,0.967892
MetroTV (2),0.0,0.756018,1.000000,0.787621,0.866532,0.721750,0.706271,0.515406,0.490598,0.0,...,0.725063,0.800008,0.725219,0.720847,0.734140,0.805991,0.704648,0.824650,0.793957,0.866532
Antara (3),0.0,0.975418,0.787621,1.000000,0.941072,0.683936,0.633230,0.488402,0.724459,0.0,...,0.928571,0.955856,0.926257,0.754000,0.859118,0.826111,0.861892,0.683763,0.900607,0.941072
Lampost (4),0.0,0.967892,0.866532,0.941072,1.000000,0.711965,0.692408,0.566717,0.753988,0.0,...,0.914931,0.951797,0.918961,0.711695,0.848781,0.776890,0.869213,0.695516,0.914830,1.000000
Detik (5),0.0,0.646393,0.721750,0.683936,0.711965,1.000000,0.985245,0.904534,0.597913,0.0,...,0.734931,0.763048,0.783943,0.933257,0.802022,0.914138,0.716264,0.804030,0.766478,0.711965
Detik (6),0.0,0.603337,0.706271,0.633230,0.692408,0.985245,1.000000,0.919935,0.595172,0.0,...,0.690719,0.731175,0.755155,0.860164,0.762231,0.867009,0.710806,0.833691,0.719901,0.692408
Detik (7),0.0,0.492366,0.515406,0.488402,0.566717,0.904534,0.919935,1.000000,0.713900,0.0,...,0.645777,0.667196,0.729959,0.760788,0.585045,0.783349,0.700774,0.636364,0.703653,0.566717
Tempo (8),0.0,0.768906,0.490598,0.724459,0.753988,0.597913,0.595172,0.713900,1.000000,0.0,...,0.872321,0.834912,0.879696,0.538166,0.501196,0.657616,0.928196,0.432666,0.866779,0.753988
Pemkab Buleleng (9),0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## **4. Analisis Spesifik (Query Style**

Disini kita akan membandingkan baris tertentu. Misal mau lihat artikel mana yang paling mirip dengan Artikel Pertama (Indeks 0 - Kemensos Evaluasi Data)

In [8]:
# Mengambil hasil similarity untuk artikel pertama saja
perbandingan_art1 = df_similarity.iloc[0]

print(f"\nAnalisis Kedekatan dengan {df_vsm.index[0]}:")
print(perbandingan_art1.sort_values(ascending=False))


Analisis Kedekatan dengan Detik (0):
Detik (0)               0.0
Detik (1)               0.0
MetroTV (2)             0.0
Antara (3)              0.0
Lampost (4)             0.0
Detik (5)               0.0
Detik (6)               0.0
Detik (7)               0.0
Tempo (8)               0.0
Pemkab Buleleng (9)     0.0
Detik (10)              0.0
Detik (11)              0.0
Detik (12)              0.0
Antara (13)             0.0
Detik (14)              0.0
Detik (15)              0.0
Detik (16)              0.0
Detik (17)              0.0
Detik (18)              0.0
Detik (19)              0.0
Detik (20)              0.0
TImes Indonesia (21)    0.0
UNESA (22)              0.0
Lampost (23)            0.0
Name: Detik (0), dtype: float64
